# S09 · Build an explainable query product

**Outcome:** Assemble a reusable query, its expected identities, evidence and user-task acceptance test.

**Time:** about 55 minutes. Run cells in order. This is the executed solution edition.

A query product has consumers and a service contract. Define the answer fields, their interpretation, graph scope, inference coverage, ordering, expected runtime, failure policy and version dependencies. A source code, a mapped concept and an inferred type should appear as different fields when users need to distinguish them.

The final query returns respiratory review records with the source encounter identifier, source code and duration. The reasoner supplies the selected review memberships; the asserted graph supplies the original evidence. The test compares exact record identifiers against the accepted source policy. An explanation template records the premises and the two schema steps. No language model is needed for the inference itself.

An AI assistant may help draft a query or explain an answer, but the application still needs parsing, an allowed query surface, execution limits, authorized graph scope, evidence checks and human review of meaning changes. Treat retrieved content as data rather than instructions. Measure whether users can identify an unsupported mapping and reproduce an answer, not only whether the generated prose sounds convincing.

In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ontology_lab").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Open this notebook from the extracted course folder.")
from ontology_lab import *
print("Course:", ROOT.name, "| source rows:", len(rows()))

Course: enterprise_ontology_tutorial | source rows: 60


## Execute the final evidence query

In [2]:
report,derived=reasoning();ds=dataset(derived)
q='''SELECT ?id ?code ?days WHERE {
 GRAPH <urn:graph:derived:v1> { ?r a ex:ReviewCandidate }
 GRAPH <urn:graph:asserted:v1> {
   ?r ex:encounterId ?id; ex:primaryCode ?code; ex:stayDays ?days
 }
} ORDER BY ?id'''
result=list(query(ds,q));display(result[:5])
assert {str(x[0]) for x in result}=={r['encounter_id'] for r in rows() if r['diag_1']=='493'}

[(rdflib.term.Literal('10555854'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/493'),
  rdflib.term.Literal('8', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('12846246'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/493'),
  rdflib.term.Literal('3', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('13056282'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/493'),
  rdflib.term.Literal('3', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('14964918'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/493'),
  rdflib.term.Literal('3', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('15996702'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/493'),
  rdflib.term.Literal('5', datatype=rdflib.term.URIRef('http://www.w3.org/200

## Attach a reproducible proof sketch

In [3]:
explanation={
 'source_fact':'record primaryCode source:493',
 'adapter_policy':'source:493 is categorized as RespiratoryCode',
 'definition':'EncounterRecord and primaryCode some RespiratoryCode -> RespiratoryCodedRecord',
 'workflow_axiom':'RespiratoryCodedRecord subclassOf ReviewCandidate',
 'claim_scope':'Record review routing only',
 'proof_kind':'Auditable proof sketch; not a minimal justification extracted from HermiT'}
display(explanation)

{'source_fact': 'record primaryCode source:493',
 'adapter_policy': 'source:493 is categorized as RespiratoryCode',
 'definition': 'EncounterRecord and primaryCode some RespiratoryCode -> RespiratoryCodedRecord',
 'workflow_axiom': 'RespiratoryCodedRecord subclassOf ReviewCandidate',
 'claim_scope': 'Record review routing only',
 'proof_kind': 'Auditable proof sketch; not a minimal justification extracted from HermiT'}

## Your turn

Return a human-use acceptance task that asks a learner to recover the original source code and distinguish a proposed mapping from an accepted one.

Replace `answer = None` with your code. A skipped exercise is reported as incomplete; it is not a pass.

In [4]:
answer = 'Find a review record, show its original source code, and explain why the proposed ICD-10-CM mapping is not yet accepted.'

In [5]:
learner_check(answer, lambda x:isinstance(x,str) and 'source' in x.lower() and 'proposed' in x.lower(), 'A usability task should test understanding of the evidence boundary.')

Exercise passed.
Out[0]: True


## Explain your model

What would you change if users consistently confuse a review candidate with a confirmed diagnosis?

Write a short answer below. Check the relevant chapter in the book before promoting a model change.

**My explanation:** Compare the proof premises, source scope and query contract described above; use your own words in a peer review.